In [2]:
"""
PROYECTO: Fine-tuning de OWLv2 para detección de personajes de One Piece
AUTOR: Tu nombre
FECHA: 2024
DESCRIPCIÓN: Este script usa OWLv2 para etiquetar imágenes automáticamente
y luego entrena un modelo YOLOv8 específico para personajes de One Piece.
"""

import os
import torch
import cv2
from PIL import Image
from transformers import Owlv2Processor, Owlv2ForObjectDetection
from autodistill.detection import CaptionOntology
from autodistill.utils import plot
from autodistill_transformers import TransformersModel
from autodistill_yolov8 import YOLOv8  # Necesitas: pip install autodistill-yolov8
import supervision as sv

# ============================================================================
# CONFIGURACIÓN INICIAL
# ============================================================================

# Crear carpetas necesarias
os.makedirs("./dataset_one_piece/images", exist_ok=True)
os.makedirs("./dataset_one_piece/labels", exist_ok=True)
os.makedirs("./modelo_entrenado", exist_ok=True)

# ============================================================================
# PASO 1: CARGAR OWLv2 (el modelo "profesor")
# ============================================================================
print("Cargando modelo OWLv2...")

# Cargar procesador y modelo
processor = Owlv2Processor.from_pretrained("google/owlv2-base-patch16-ensemble")
model = Owlv2ForObjectDetection.from_pretrained("google/owlv2-base-patch16-ensemble")

# Función de inferencia personalizada para OWLv2
def inference_owlv2(image, prompts):
    """
    Ejecuta OWLv2 en una imagen con los prompts dados
    
    Args:
        image: Imagen PIL
        prompts: Lista de textos a buscar
    
    Returns:
        Resultados con bounding boxes
    """
    inputs = processor(text=prompts, images=image, return_tensors="pt")
    outputs = model(**inputs)
    
    target_sizes = torch.Tensor([image.size[::-1]])
    
    results = processor.post_process_object_detection(
        outputs=outputs, target_sizes=target_sizes, threshold=0.05  # Umbral bajo para capturar todo
    )[0]
    
    return results

# ============================================================================
# PASO 2: DEFINIR QUÉ QUEREMOS DETECTAR (ONTOLOGÍA)
# ============================================================================
# Mapeo entre lo que buscas y la etiqueta final
# IMPORTANTE: Prueba diferentes descripciones en inglés (funciona mejor)
ontology = CaptionOntology({
    # Descripción en inglés (lo que entiende OWLv2) : Etiqueta final
    "a photo of Monkey D. Luffy, anime character with straw hat": "Luffy",
    "a photo of Roronoa Zoro, anime character with green hair and swords": "Zoro", 
    "a photo of Vinsmoke Sanji, anime character with blonde hair and black suit": "Sanji",
    "a photo of Nami, anime character with orange hair": "Nami",
    "a photo of Usopp, anime character with long nose": "Usopp",
    "a photo of Tony Tony Chopper, small reindeer anime character": "Chopper",
    "a photo of Nico Robin, anime character with black hair": "Robin",
    "a photo of Franky, anime character with blue hair and robot body": "Franky",
    "a photo of Brook, skeleton anime character with afro": "Brook",
    "a photo of Jinbe, fishman anime character": "Jinbe"
})

print("Ontología definida:")
for prompt, label in ontology.prompts.items():
    print(f"  Buscar: '{prompt}' -> Etiqueta: '{label}'")

# ============================================================================
# PASO 3: CREAR MODELO BASE (EL QUE ETIQUETARÁ)
# ============================================================================
base_model = TransformersModel(
    ontology=ontology,
    callback=inference_owlv2,
)

# ============================================================================
# PASO 4: ETIQUETAR AUTOMÁTICAMENTE TUS IMÁGENES
# ============================================================================
print("\n" + "="*60)
print("PASO 4: Etiquetando imágenes automáticamente...")
print("="*60)

# IMPORTANTE: Coloca tus imágenes de One Piece en esta carpeta
# Crea la carpeta 'imagenes_one_piece' y pon allí todas tus fotos
ruta_imagenes = "./imagenes_one_piece"  # CAMBIA ESTO a tu carpeta con imágenes

# Verificar que la carpeta existe
if not os.path.exists(ruta_imagenes):
    print(f"⚠️  La carpeta {ruta_imagenes} no existe. Creándola...")
    os.makedirs(ruta_imagenes)
    print(f"👉 Por favor, coloca tus imágenes de One Piece en: {ruta_imagenes}")
    print("   Luego ejecuta este script nuevamente.")
    exit()

# Contar imágenes
imagenes = [f for f in os.listdir(ruta_imagenes) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"📸 Se encontraron {len(imagenes)} imágenes para procesar")

# Etiquetar todas las imágenes (esto generará archivos .txt con las anotaciones)
# Las anotaciones se guardan en la misma carpeta en formato YOLO
base_model.label(
    input_folder=ruta_imagenes,      # Carpeta con imágenes
    extension=".jpg",                 # Extensión de las imágenes
    output_folder="./dataset_one_piece"  # Carpeta donde guardar el dataset
)

print(f"✅ Etiquetado completado. Las anotaciones están en ./dataset_one_piece")

# ============================================================================
# PASO 5: VISUALIZAR ALGUNAS ANOTACIONES (para verificar)
# ============================================================================
print("\n" + "="*60)
print("PASO 5: Verificando anotaciones...")
print("="*60)

# Cargar una imagen de ejemplo
imagenes_procesadas = [f for f in os.listdir("./dataset_one_piece/images") if f.endswith('.jpg')]
if imagenes_procesadas:
    img_ejemplo = imagenes_procesadas[0]
    image_path = f"./dataset_one_piece/images/{img_ejemplo}"
    label_path = f"./dataset_one_piece/labels/{img_ejemplo.replace('.jpg', '.txt')}"
    
    if os.path.exists(label_path):
        print(f"✅ Anotaciones encontradas para {img_ejemplo}")
        
        # Leer anotaciones
        with open(label_path, 'r') as f:
            anotaciones = f.readlines()
        
        print(f"   {len(anotaciones)} objetos detectados:")
        for ann in anotaciones[:5]:  # Mostrar primeras 5
            clase, x, y, w, h = ann.strip().split()
            print(f"   - Clase {clase} en centro=({x}, {y}) tamaño=({w}, {h})")
    else:
        print(f"⚠️  No hay anotaciones para {img_ejemplo}")
else:
    print("⚠️  No se encontraron imágenes procesadas")

# ============================================================================
# PASO 6: ENTRENAR MODELO NUEVO (YOLOv8)
# ============================================================================
print("\n" + "="*60)
print("PASO 6: Entrenando modelo YOLOv8 con tus datos...")
print("="*60)

# Crear modelo objetivo (el que vamos a entrenar)
target_model = YOLOv8("yolov8n.pt")  # Usamos YOLOv8 nano (rápido)

# Entrenar
print("🎯 Comenzando entrenamiento... (esto puede tomar varios minutos)")
target_model.train(
    "./dataset_one_piece",  # Carpeta con el dataset
    epochs=50,              # Número de épocas (ajusta según necesites)
    batch=8,                 # Tamaño del batch (reduce si tienes poca memoria)
    imgsz=640,               # Tamaño de imagen
    patience=10              # Detener si no mejora después de 10 épocas
)

print("✅ Entrenamiento completado!")

# ============================================================================
# PASO 7: GUARDAR MODELO ENTRENADO
# ============================================================================
target_model.model.save("./modelo_entrenado/modelo_one_piece.pt")
print(f"📦 Modelo guardado en: ./modelo_entrenado/modelo_one_piece.pt")

# ============================================================================
# PASO 8: PROBAR EL MODELO ENTRENADO
# ============================================================================
print("\n" + "="*60)
print("PASO 8: Probando el modelo entrenado...")
print("="*60)

# Cargar modelo entrenado
from ultralytics import YOLO
modelo_final = YOLO("./modelo_entrenado/modelo_one_piece.pt")

# Probar con una imagen nueva (si existe)
if imagenes_procesadas:
    img_test = f"./dataset_one_piece/images/{imagenes_procesadas[0]}"
    results = modelo_final(img_test)
    
    # Mostrar resultados
    print(f"Resultados para {imagenes_procesadas[0]}:")
    for r in results:
        boxes = r.boxes
        for box in boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            # Obtener nombre de la clase
            nombre_clase = list(ontology.prompts.values())[cls] if cls < len(ontology.prompts) else f"Clase_{cls}"
            print(f"   • {nombre_clase}: confianza {conf:.3f}")
    
    # Guardar imagen con detecciones
    resultados_img = results[0].plot()
    cv2.imwrite("./resultado_prueba.jpg", resultados_img)
    print("📸 Imagen con detecciones guardada como 'resultado_prueba.jpg'")

print("\n" + "="*60)
print("🎉 ¡PROCESO COMPLETADO!")
print("="*60)
print("\nAhora tienes:")
print("  • Un modelo YOLOv8 personalizado para personajes de One Piece")
print("  • Guardado en: ./modelo_entrenado/modelo_one_piece.pt")
print("\nPara usarlo en el futuro:")
print("  from ultralytics import YOLO")
print("  modelo = YOLO('./modelo_entrenado/modelo_one_piece.pt')")
print("  resultados = modelo('tu_imagen.jpg')")

ModuleNotFoundError: No module named 'cv2'

In [3]:
"""
PROYECTO: Usar modelo entrenado para detectar personajes de One Piece
"""

from ultralytics import YOLO
import cv2
from PIL import Image
import matplotlib.pyplot as plt

# Cargar modelo entrenado
modelo = YOLO("./modelo_entrenado/modelo_one_piece.pt")

# Diccionario de clases (debe coincidir con tu ontología)
clases = {
    0: "Luffy", 1: "Zoro", 2: "Sanji", 3: "Nami", 4: "Usopp",
    5: "Chopper", 6: "Robin", 7: "Franky", 8: "Brook", 9: "Jinbe"
}

def detectar_personajes(ruta_imagen, confianza=0.25):
    """
    Detecta personajes de One Piece en una imagen
    
    Args:
        ruta_imagen: Ruta a la imagen
        confianza: Umbral de confianza (0-1)
    
    Returns:
        Imagen con detecciones dibujadas
    """
    # Ejecutar detección
    results = modelo(ruta_imagen, conf=confianza)
    
    # Procesar resultados
    imagen = cv2.imread(ruta_imagen)
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    
    for r in results:
        boxes = r.boxes
        for box in boxes:
            # Obtener coordenadas
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            
            # Obtener clase y confianza
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            nombre = clases.get(cls, f"Desconocido_{cls}")
            
            # Dibujar rectángulo
            cv2.rectangle(imagen, (x1, y1), (x2, y2), (255, 0, 0), 2)
            
            # Dibujar etiqueta
            etiqueta = f"{nombre} {conf:.2f}"
            cv2.putText(imagen, etiqueta, (x1, y1-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
            
            print(f"✅ Detectado: {nombre} con confianza {conf:.3f}")
    
    return imagen

# ============================================================================
# EJEMPLO DE USO
# ============================================================================
if __name__ == "__main__":
    # Prueba con una imagen
    imagen_resultado = detectar_personajes("tu_imagen_one_piece.jpg", confianza=0.25)
    
    # Mostrar resultado
    plt.imshow(imagen_resultado)
    plt.axis('off')
    plt.show()
    
    # Guardar resultado
    imagen_resultado_bgr = cv2.cvtColor(imagen_resultado, cv2.COLOR_RGB2BGR)
    cv2.imwrite("deteccion_one_piece.jpg", imagen_resultado_bgr)

ModuleNotFoundError: No module named 'ultralytics'